In [13]:
# Cell 1 - Configurar rutas
WORKSPACE_ID = "956e9e93-b6e5-451c-b114-91929dbfe82c"
BRONZE_ID = "702e246c-b575-416c-b603-8c63a947e802"
SILVER_ID = "4bb699f7-bd8e-4f14-a3bc-5585f5067393"

BRONZE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_ID}"
SILVER_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{SILVER_ID}"

print("Rutas configuradas OK")
print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")

StatementMeta(, 15fa6ed3-759e-4075-b2bf-b16783e2eae4, 15, Finished, Available, Finished, False)

Rutas configuradas OK
Bronze: abfss://956e9e93-b6e5-451c-b114-91929dbfe82c@onelake.dfs.fabric.microsoft.com/702e246c-b575-416c-b603-8c63a947e802
Silver: abfss://956e9e93-b6e5-451c-b114-91929dbfe82c@onelake.dfs.fabric.microsoft.com/4bb699f7-bd8e-4f14-a3bc-5585f5067393


In [14]:
# Cell 2 - Leer tablas Bronze y limpiar para Silver
from pyspark.sql.functions import col, trim, upper, when, lit
from pyspark.sql.types import IntegerType, DoubleType, DateType

TABLES = {
    "Sales": [
        "SalesOrderHeader",
        "SalesOrderDetail",
        "Customer",
        "SalesTerritory",
        "SalesPerson"
    ],
    "Production": [
        "Product",
        "ProductCategory",
        "ProductSubcategory",
        "ProductInventory",
        "WorkOrder"
    ],
    "Purchasing": [
        "PurchaseOrderHeader",
        "Vendor"
    ]
}

YEARS = ["2019", "2022", "2025"]

for schema, tables in TABLES.items():
    for table in tables:
        dfs = []
        for year in YEARS:
            bronze_table = f"bronze_{schema}_{table}_{year}".lower()
            path = f"{BRONZE_PATH}/Tables/dbo/{bronze_table}"
            try:
                df = spark.read.format("delta").load(path)
                dfs.append(df)
            except Exception as e:
                print(f"ERROR leyendo {bronze_table}: {e}")

        if dfs:
            # Unificar las 3 versiones
            df_union = dfs[0]
            for df in dfs[1:]:
                df_union = df_union.unionByName(df, allowMissingColumns=True)

            # Limpiar columnas string (trim espacios)
            string_cols = [f.name for f in df_union.schema.fields if str(f.dataType) == "StringType()"]
            for c in string_cols:
                df_union = df_union.withColumn(c, trim(col(c)))

            # Guardar en Silver
            silver_path = f"{SILVER_PATH}/Tables/silver_{schema}_{table}"
            df_union.write.format("delta").mode("overwrite").save(silver_path)
            
            print(f"OK: silver_{schema}_{table} ({df_union.count():,} filas)")

StatementMeta(, 15fa6ed3-759e-4075-b2bf-b16783e2eae4, 16, Finished, Available, Finished, False)

OK: silver_Sales_SalesOrderHeader (94,395 filas)
OK: silver_Sales_SalesOrderDetail (363,951 filas)
OK: silver_Sales_Customer (59,460 filas)
OK: silver_Sales_SalesTerritory (30 filas)
OK: silver_Sales_SalesPerson (51 filas)
OK: silver_Production_Product (1,512 filas)
OK: silver_Production_ProductCategory (12 filas)
OK: silver_Production_ProductSubcategory (111 filas)
OK: silver_Production_ProductInventory (3,207 filas)
OK: silver_Production_WorkOrder (217,773 filas)
OK: silver_Purchasing_PurchaseOrderHeader (12,036 filas)
OK: silver_Purchasing_Vendor (312 filas)


In [15]:
# Cell 3 - Verificación Silver
from pyspark.sql.functions import col

# Verificar que la columna source_year tiene las 3 versiones
df = spark.read.format("delta").load(f"{SILVER_PATH}/Tables/silver_Sales_SalesOrderHeader")

print("Filas por año en SalesOrderHeader:")
df.groupBy("source_year").count().orderBy("source_year").show()

print(f"Total columnas: {len(df.columns)}")
print(f"Columnas: {df.columns}")

StatementMeta(, 15fa6ed3-759e-4075-b2bf-b16783e2eae4, 17, Finished, Available, Finished, False)

Filas por año en SalesOrderHeader:
+-----------+-----+
|source_year|count|
+-----------+-----+
|       2019|31465|
|       2022|31465|
|       2025|31465|
+-----------+-----+

Total columnas: 27
Columnas: ['SalesOrderID', 'RevisionNumber', 'OrderDate', 'DueDate', 'ShipDate', 'Status', 'OnlineOrderFlag', 'SalesOrderNumber', 'PurchaseOrderNumber', 'AccountNumber', 'CustomerID', 'SalesPersonID', 'TerritoryID', 'BillToAddressID', 'ShipToAddressID', 'ShipMethodID', 'CreditCardID', 'CreditCardApprovalCode', 'CurrencyRateID', 'SubTotal', 'TaxAmt', 'Freight', 'TotalDue', 'Comment', 'rowguid', 'ModifiedDate', 'source_year']


In [2]:
# Cell 2 - Leer todos los parquet y registrarlos como tablas Delta en Bronze
from pyspark.sql.functions import lit

# Configuración de rutas
TABLES = {
    "Sales": [
        "SalesOrderHeader",
        "SalesOrderDetail",
        "Customer",
        "SalesTerritory",
        "SalesPerson"
    ],
    "Production": [
        "Product",
        "ProductCategory",
        "ProductSubcategory",
        "ProductInventory",
        "WorkOrder"
    ],
    "Purchasing": [
        "PurchaseOrderHeader",
        "Vendor"
    ]
}

YEARS = {
    "AW2019": "2019",
    "AW2022": "2022",
    "AW2025": "2025"
}

# Leer y guardar cada tabla como Delta
for folder, year in YEARS.items():
    for schema, tables in TABLES.items():
        for table in tables:
            path = f"Files/{folder}/AW_{schema}_{year}/{schema}/{table}.parquet"
            
            try:
                df = spark.read.parquet(path)
                
                # Agregar columna de año para identificar la fuente
                df = df.withColumn("source_year", lit(year))
                
                # Nombre de tabla en el Lakehouse: bronze_Sales_SalesOrderHeader_2019
                table_name = f"bronze_{schema}_{table}_{year}"
                
                df.write.format("delta").mode("overwrite").saveAsTable(table_name)
                
                print(f"OK: {table_name} ({df.count():,} filas)")
                
            except Exception as e:
                print(f"ERROR: {schema}.{table} {year} -> {e}")

StatementMeta(, 361b8e56-5f82-474b-be3c-23875eebf3c9, 4, Finished, Available, Finished, False)

OK: bronze_Sales_SalesOrderHeader_2019 (31,465 filas)
OK: bronze_Sales_SalesOrderDetail_2019 (121,317 filas)
OK: bronze_Sales_Customer_2019 (19,820 filas)
OK: bronze_Sales_SalesTerritory_2019 (10 filas)
OK: bronze_Sales_SalesPerson_2019 (17 filas)
OK: bronze_Production_Product_2019 (504 filas)
OK: bronze_Production_ProductCategory_2019 (4 filas)
OK: bronze_Production_ProductSubcategory_2019 (37 filas)
OK: bronze_Production_ProductInventory_2019 (1,069 filas)
OK: bronze_Production_WorkOrder_2019 (72,591 filas)
OK: bronze_Purchasing_PurchaseOrderHeader_2019 (4,012 filas)
OK: bronze_Purchasing_Vendor_2019 (104 filas)
OK: bronze_Sales_SalesOrderHeader_2022 (31,465 filas)
OK: bronze_Sales_SalesOrderDetail_2022 (121,317 filas)
OK: bronze_Sales_Customer_2022 (19,820 filas)
OK: bronze_Sales_SalesTerritory_2022 (10 filas)
OK: bronze_Sales_SalesPerson_2022 (17 filas)
OK: bronze_Production_Product_2022 (504 filas)
OK: bronze_Production_ProductCategory_2022 (4 filas)
OK: bronze_Production_ProductS

In [1]:
# Cell 3 - Verificación final
tables = spark.catalog.listTables()
print(f"Total tablas en LH_Bronze: {len(tables)}")
print("\nEjemplo - SalesOrderHeader 2019 vs 2025:")
df_2019 = spark.table("bronze_Sales_SalesOrderHeader_2019")
df_2025 = spark.table("bronze_Sales_SalesOrderHeader_2025")
print(f"2019: {df_2019.count():,} filas | columnas: {len(df_2019.columns)}")
print(f"2025: {df_2025.count():,} filas | columnas: {len(df_2025.columns)}")

StatementMeta(, 15fa6ed3-759e-4075-b2bf-b16783e2eae4, 3, Finished, Available, Finished, False)

Total tablas en LH_Bronze: 36

Ejemplo - SalesOrderHeader 2019 vs 2025:
2019: 31,465 filas | columnas: 27
2025: 31,465 filas | columnas: 27
